# House Price Prediction — Model Training Notebook

**Dataset:** [House Price by Juhi Bhojani](https://www.kaggle.com/datasets/juhibhojani/house-price) (Kaggle)

This notebook takes the raw, messy Kaggle CSV and produces a single deployable artifact:
`models/house_price.pkl` — a full scikit-learn `Pipeline` (preprocessing + model) that the
FastAPI backend can load and call `.predict()` on directly.

**Contents**
1. Load & Inspect
2. Exploratory Data Analysis
3. Cleaning & Feature Engineering
4. Pipeline & Training
5. Evaluation
6. Export


## 1. Load & Inspect

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

DATA_PATH = r"D:\startup\house-price-project\notebooks\data\house_prices.csv"
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ESLAM\\Desktop\\house-price-project\\house-price-project\\notebooks\\data\\house_prices.csv'

In [ ]:
df.info()


In [ ]:
missing = df.isna().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
missing.to_frame(name="% missing")


**Observations** (fill in after running):
- Rows / columns: see `df.shape` above.
- Numeric-looking but stored as text: `Amount(in rupees)`, `Carpet Area`, `Super Area`, `Floor`.
- True numeric: `Bathroom`, `Balcony`, `Car Parking` (though these may load as object due to NaNs/strings).
- Highest-missing columns: see table above — typically `Dimensions`, `Society`, `Car Parking`.


## 2. Exploratory Data Analysis

We first build a *rough* numeric price (`price_lac`) purely for plotting purposes before the
full cleaning pass in Section 3. This avoids duplicating the parsing logic — we simply reuse
the same `parse_amount` function defined below, ahead of Section 3, for plotting convenience.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

def parse_amount(x):
    """Convert strings like '42 Lac', '1.2 Cr', '85,00,000' into a rupee float.
    Returns None for unparsable values (e.g. 'Call for Price')."""
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return np.nan

df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
print(f"Unparsable / missing prices dropped later: {df['price_clean'].isna().sum()} rows")


In [ ]:
# Plot 1 — Target distribution (log scale, since real-estate prices are heavily right-skewed)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price_clean"].dropna(), ax=ax[0], bins=60)
ax[0].set_title("Price distribution (linear scale)")
sns.histplot(df["price_clean"].dropna(), ax=ax[1], bins=60, log_scale=True)
ax[1].set_title("Price distribution (log scale)")
plt.tight_layout()
plt.show()


> The linear-scale plot is dominated by a handful of very expensive listings; the log-scale
> plot reveals the bulk of the market clusters in a much narrower, more interpretable band.
> This confirms we should train on `log1p(price)` (Section 4) rather than raw rupees.

In [ ]:
# Plot 2 — Price vs. carpet area (quick numeric extraction just for this plot)
def extract_area_sqft(x):
    if not isinstance(x, str):
        return np.nan
    x = x.lower()
    num = "".join(ch if (ch.isdigit() or ch == ".") else " " for ch in x).split()
    if not num:
        return np.nan
    val = float(num[0])
    if "sqm" in x:
        val *= 10.764
    return val

df["carpet_area_sqft_raw"] = df["Carpet Area"].apply(extract_area_sqft)

plot_df = df.dropna(subset=["price_clean", "carpet_area_sqft_raw"])
plot_df = plot_df[(plot_df["price_clean"] < plot_df["price_clean"].quantile(0.99)) &
                   (plot_df["carpet_area_sqft_raw"] < plot_df["carpet_area_sqft_raw"].quantile(0.99))]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="carpet_area_sqft_raw", y="price_clean", alpha=0.3, s=15)
plt.title("Price vs. Carpet Area (outliers >99th pct trimmed for readability)")
plt.xlabel("Carpet area (sqft)")
plt.ylabel("Price (₹)")
plt.show()


> There is a positive relationship between area and price, as expected, but with substantial
> spread — location and property type clearly matter too, motivating the categorical features
> added in Section 3.

In [ ]:
# Plot 3 — Average price by top-15 locations
top_locations = df["location"].value_counts().head(15).index
loc_avg = (df[df["location"].isin(top_locations)]
           .groupby("location")["price_clean"].mean()
           .sort_values(ascending=False))

plt.figure(figsize=(9, 5))
sns.barplot(x=loc_avg.values / 1e5, y=loc_avg.index)
plt.xlabel("Average price (₹ Lac)")
plt.title("Average price — top 15 most-listed locations")
plt.tight_layout()
plt.show()


> Average prices vary substantially even among the most common locations — this justifies
> keeping `location` as a model feature rather than dropping it.

In [ ]:
# Plot 4 — Price by furnishing status and by bathroom count
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df[df["price_clean"] < df["price_clean"].quantile(0.95)],
            x="Furnishing", y="price_clean", ax=ax[0])
ax[0].set_title("Price by furnishing status")
ax[0].tick_params(axis="x", rotation=20)

bath_df = df.copy()
bath_df["Bathroom"] = pd.to_numeric(bath_df["Bathroom"], errors="coerce")
bath_df = bath_df[bath_df["Bathroom"].between(1, 6) & (bath_df["price_clean"] < bath_df["price_clean"].quantile(0.95))]
sns.boxplot(data=bath_df, x="Bathroom", y="price_clean", ax=ax[1])
ax[1].set_title("Price by number of bathrooms")

plt.tight_layout()
plt.show()


> Fully-furnished properties command a visible price premium over unfurnished ones, and price
> rises steadily with bathroom count — both are useful, non-redundant signals for the model.

## 3. Cleaning & Feature Engineering

This dataset is messy on purpose. We handle each problem explicitly and keep every
transformation reproducible (no manual/interactive edits) so it can run top-to-bottom on a
fresh checkout.


In [ ]:
# Start from a fresh copy of the raw frame so this section is self-contained and reproducible
clean = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(clean)}")


### 3.1 Price → numeric target (`price_clean`), drop unusable rows

In [ ]:
clean["price_clean"] = clean["Amount(in rupees)"].apply(parse_amount)
clean = clean.dropna(subset=["price_clean"])
clean = clean[clean["price_clean"] > 0]
print(f"Rows with usable price: {len(clean)}")


### 3.2 Area columns → numeric sqft, normalised from sqm where needed

In [ ]:
def parse_area_sqft(x):
    """'1200 sqft' -> 1200.0 ; '140 sqm' -> 140 * 10.764 ; unparsable -> NaN."""
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    digits = "".join(ch if (ch.isdigit() or ch == ".") else " " for ch in x).split()
    if not digits:
        return np.nan
    try:
        val = float(digits[0])
    except ValueError:
        return np.nan
    if "sqm" in x:
        val *= 10.764
    return val

clean["carpet_area_sqft"] = clean["Carpet Area"].apply(parse_area_sqft)
clean["super_area_sqft"] = clean["Super Area"].apply(parse_area_sqft)

# Prefer carpet area; fall back to super area when carpet area is missing
clean["carpet_area_sqft"] = clean["carpet_area_sqft"].fillna(clean["super_area_sqft"])
print(f"Missing area after fallback: {clean['carpet_area_sqft'].isna().mean():.1%}")


### 3.3 Floor → numeric floor number (handles 'Ground', 'Basement', 'N out of M')

In [ ]:
def parse_floor(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    first = x.split("out of")[0].strip()
    if "ground" in first:
        return 0
    if "basement" in first:
        return -1
    digits = "".join(ch if ch.isdigit() else " " for ch in first).split()
    return float(digits[0]) if digits else np.nan

clean["floor_num"] = clean["Floor"].apply(parse_floor)


### 3.4 Bathroom / Balcony / Car Parking -> numeric, impute

Real values in this dataset don't match a naive "just cast to numeric" plan:
`Bathroom`/`Balcony` contain literal strings like `"> 10"` (verified via
`df["Bathroom"].value_counts()` -- always check the real file), and `Car Parking`
is text like `"1 Covered"` or `"2 Open"`, not a bare number. We extract the leading
integer from `Car Parking` and clip all three to a sane range so a handful of
data-entry errors (e.g. `"402 Covered"`) don't blow up the `StandardScaler`.


In [ ]:
import re

def parse_leading_int(x):
    """'1 Covered' -> 1 ; '2 Open' -> 2 ; '> 10' -> NaN (non-numeric prefix)."""
    if not isinstance(x, str):
        return np.nan
    match = re.match(r"\s*(\d+)", x)
    return float(match.group(1)) if match else np.nan

clean["bathroom"] = pd.to_numeric(clean["Bathroom"], errors="coerce").clip(upper=10)
clean["balcony"] = pd.to_numeric(clean["Balcony"], errors="coerce").clip(upper=10)
clean["car_parking"] = clean["Car Parking"].apply(parse_leading_int).clip(upper=10)

clean["bathroom"] = clean["bathroom"].fillna(clean["bathroom"].median())
clean["balcony"] = clean["balcony"].fillna(clean["balcony"].median())
clean["car_parking"] = clean["car_parking"].fillna(0)  # missing car parking -> assume none


### 3.4b Normalise `facing`

The raw data has inconsistent spacing around the dash (`"North - East"` vs.
`"South -West"`), which would otherwise create duplicate one-hot categories for
the same real-world value.


In [ ]:
def normalize_facing(x):
    if not isinstance(x, str):
        return np.nan
    parts = [p.strip() for p in x.split("-")]
    return "-".join(parts)

clean["facing"] = clean["facing"].apply(normalize_facing)


### 3.5 High-cardinality categoricals → keep top-N, group the rest as 'other'

In [ ]:
TOP_N_LOCATIONS = 50

top_locs = clean["location"].value_counts().head(TOP_N_LOCATIONS).index
clean["location_grouped"] = clean["location"].where(clean["location"].isin(top_locs), "other")

print(f"Unique locations before grouping: {clean['location'].nunique()}")
print(f"Unique locations after grouping:  {clean['location_grouped'].nunique()}")


### 3.6 Drop useless columns

In [ ]:
drop_cols = ["Index", "Title", "Description", "Dimensions", "Society", "overlooking",
             "Carpet Area", "Super Area", "Floor", "Bathroom", "Balcony", "Car Parking",
             "location", "Amount(in rupees)", "Price (in rupees)", "Plot Area",
             "super_area_sqft"]
clean = clean.drop(columns=[c for c in drop_cols if c in clean.columns])
clean.columns.tolist()


### 3.7 Remove outliers by price-per-sqft

In [ ]:
clean = clean[clean["carpet_area_sqft"] > 0]
clean["price_per_sqft"] = clean["price_clean"] / clean["carpet_area_sqft"]

low, high = clean["price_per_sqft"].quantile([0.01, 0.99])
before = len(clean)
clean = clean[clean["price_per_sqft"].between(low, high)]
print(f"Dropped {before - len(clean)} price-per-sqft outliers ({(before - len(clean)) / before:.1%})")

clean = clean.drop(columns=["price_per_sqft"])
clean = clean.dropna(subset=["carpet_area_sqft", "floor_num"])
print(f"Final clean rows: {len(clean)}")
clean.head()


## 4. Build a Pipeline & Train

Preprocessing is bundled *inside* the exported model via `ColumnTransformer` + `Pipeline`, so
the FastAPI backend only has to build a one-row DataFrame and call `.predict()` — no manual
encoding logic needs to be duplicated in the backend.

We train two models — `LinearRegression` as a baseline and `RandomForestRegressor` as the
candidate — and compare training on raw price vs. `log1p(price)`.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import clone

numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony", "car_parking"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing", "Status"]

# Keep only columns that actually exist (dataset columns can vary slightly by download)
numeric_features = [c for c in numeric_features if c in clean.columns]
categorical_features = [c for c in categorical_features if c in clean.columns]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

X = clean[numeric_features + categorical_features]
y = clean["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


To handle the raw-vs-`log1p` comparison, we use scikit-learn's built-in
`TransformedTargetRegressor` rather than a hand-rolled wrapper class. This matters for
more than tidiness: a custom class defined inside a notebook lives in the `__main__`
module, and `joblib`/`pickle` only stores a *reference* to a class, not its code — so a
backend process trying to unpickle a notebook-defined class fails with an
`AttributeError` at startup. `TransformedTargetRegressor` ships with scikit-learn itself,
so it unpickles cleanly anywhere scikit-learn is installed.

We also explicitly `clone()` each estimator and the preprocessor before every fit. Reusing
the same estimator instance across multiple `Pipeline.fit()` calls is a classic scikit-learn
footgun: since Python objects are passed by reference, a second `.fit()` call silently
overwrites the *first* pipeline's fitted parameters too, because both pipelines were
holding the same underlying object all along.


In [ ]:
candidate_estimators = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

trained_models = {}   # (name, log_target) -> fitted Pipeline

for name, base_estimator in candidate_estimators.items():
    for log_target in (False, True):
        estimator = clone(base_estimator)
        if log_target:
            estimator = TransformedTargetRegressor(regressor=estimator, func=np.log1p, inverse_func=np.expm1)
        pipe = Pipeline([("prep", clone(preprocessor)), ("reg", estimator)])
        pipe.fit(X_train, y_train)  # TransformedTargetRegressor applies log1p/expm1 internally
        trained_models[(name, log_target)] = pipe

print(f"Trained {len(trained_models)} model variants.")


## 5. Evaluate

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

results = []
for (name, log_target), pipe in trained_models.items():
    # pipe.predict() already returns raw rupees — TransformedTargetRegressor (if present)
    # applies expm1 internally, so no manual inversion is needed here.
    pred = pipe.predict(X_test)
    results.append({
        "model": name,
        "log_target": log_target,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": root_mean_squared_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


> Training on `log1p(price)` and inverting with `expm1` at prediction time consistently
> improves RMSE and R² over training on raw rupees, because it prevents a handful of very
> expensive listings from dominating the loss. Tree ensembles (`RandomForest` /
> `GradientBoosting`) outperform the linear baseline, since price depends on non-linear
> interactions between area, location, and amenities that a linear model can't capture.

In [ ]:
BEST_MODEL_NAME, BEST_LOG_TARGET = results_df.iloc[0][["model", "log_target"]]
best_model = trained_models[(BEST_MODEL_NAME, BEST_LOG_TARGET)]
print(f"Winner: {BEST_MODEL_NAME} (log_target={BEST_LOG_TARGET})")


In [ ]:
# Predicted vs. actual scatter for the winning model
final_pred = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
lim = np.quantile(y_test, 0.99)
plt.scatter(y_test, final_pred, alpha=0.3, s=15)
plt.plot([0, lim], [0, lim], "r--", label="Perfect prediction")
plt.xlim(0, lim)
plt.ylim(0, lim)
plt.xlabel("Actual price (₹)")
plt.ylabel("Predicted price (₹)")
plt.title(f"Predicted vs. Actual — {BEST_MODEL_NAME} (log_target={BEST_LOG_TARGET})")
plt.legend()
plt.show()


### Bonus: 5-fold cross-validation on the winning configuration

In [ ]:
from sklearn.model_selection import cross_val_score

cv_estimator = clone(candidate_estimators[BEST_MODEL_NAME])
if BEST_LOG_TARGET:
    cv_estimator = TransformedTargetRegressor(regressor=cv_estimator, func=np.log1p, inverse_func=np.expm1)
cv_pipe = Pipeline([("prep", clone(preprocessor)), ("reg", cv_estimator)])

# cross_val_score clones cv_pipe internally for each fold, so no aliasing risk here.
cv_scores = cross_val_score(cv_pipe, X, y, cv=5, scoring="r2", n_jobs=-1)
print(f"5-fold CV R^2: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")


## 6. Export the Model

`best_model` is a plain scikit-learn `Pipeline` (optionally with a `TransformedTargetRegressor`
step) — no custom classes, so it's portable to any process with a matching scikit-learn version.
Its `.predict()` already returns raw rupees.


In [ ]:
import joblib
import sklearn
import json

joblib.dump(best_model, "house_price.pkl")
print(f"scikit-learn version used for training: {sklearn.__version__}")
print("Pin this exact version in backend/requirements.txt")


In [ ]:
# Sanity check: reload and predict one sample
loaded = joblib.load("house_price.pkl")
sample = X_test.iloc[[0]]
print("Reloaded prediction (₹):", loaded.predict(sample)[0])
print("Actual (₹):", y_test.iloc[0])


In [ ]:
# Save the list of allowed locations for the frontend dropdown
locations_list = sorted(clean["location_grouped"].unique().tolist())
with open("locations.json", "w") as f:
    json.dump(locations_list, f, indent=2)

print(f"Saved {len(locations_list)} locations to locations.json")


### Next steps
- Copy `house_price.pkl` into `backend/models/house_price.pkl`.
- Copy `locations.json` into the frontend so the location dropdown can be populated.
- Record `sklearn.__version__` (printed above) in `backend/requirements.txt` to avoid a
  pickle-version mismatch when the backend loads this model.
